# RoboReviews — Model Evaluation

This notebook is the second deliverable. It evaluates each of the three models in the pipeline using the metrics the brief asks for:

- **Classifier** — Accuracy, Precision, Recall, F1 (macro), confusion matrix
- **Clusterer** — silhouette score across the candidate `k` range, cluster sizes, sample products per cluster
- **Summarizer** — ROUGE-1 / ROUGE-2 / ROUGE-L against the deterministic reference, plus length & uniqueness sanity checks

Sections:
1. Setup
2. Load and split data
3. Sentiment classifier — metrics, confusion matrix, failure inspection
4. Clusterer — silhouette curve, cluster sizes, qualitative inspection
5. Summarizer — ROUGE vs. deterministic reference
6. Failure-case notes
7. Takeaways

## 1. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    silhouette_score,
)
from sklearn.model_selection import train_test_split

from src.clustering import ProductClusterer
from src.config import MAX_CLUSTERS, MIN_CLUSTERS, PRIMARY_DATASET, RAW_DATA_DIR
from src.preprocessing import load_reviews_csv
from src.sentiment import SentimentAnalyzer, sentiment_from_rating
from src.summarization import RecommendationWriter, build_safe_prompt, load_summarizer

RANDOM_STATE = 42
LABELS = ["negative", "neutral", "positive"]

## 2. Load and split data

We hold out a stratified 20% slice of the cleaned dataset for evaluation. The classifier never sees the rating; the rating is used only to derive the ground-truth label.

In [ ]:
df = load_reviews_csv(RAW_DATA_DIR / PRIMARY_DATASET)
df["sentiment_true"] = df["reviews.rating"].map(sentiment_from_rating)

# Stratified split so every class is represented in the eval slice.
_, eval_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["sentiment_true"],
    random_state=RANDOM_STATE,
)

# A 1k-row eval cap keeps the notebook responsive on a laptop. Remove for full eval.
EVAL_N = min(1000, len(eval_df))
eval_df = eval_df.sample(n=EVAL_N, random_state=RANDOM_STATE).reset_index(drop=True)
print(f"Eval slice: {len(eval_df)} reviews")
eval_df["sentiment_true"].value_counts(normalize=True).round(3)

## 3. Sentiment classifier evaluation

Predictions come from `distilbert-base-uncased-finetuned-sst-2-english` running on the review text. Low-confidence predictions are mapped to `neutral` (see `NEUTRAL_CONFIDENCE_THRESHOLD` in `src/config.py`).

In [ ]:
analyzer = SentimentAnalyzer()
preds = analyzer.predict_texts(eval_df["reviews.text"].tolist())
eval_df["sentiment_pred"] = preds

In [ ]:
report = classification_report(
    eval_df["sentiment_true"],
    eval_df["sentiment_pred"],
    labels=LABELS,
    digits=3,
    zero_division=0,
)
print(report)

In [ ]:
cm = confusion_matrix(eval_df["sentiment_true"], eval_df["sentiment_pred"], labels=LABELS)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(LABELS)), LABELS)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("Predicted")
ax.set_ylabel("True (rating-derived)")
ax.set_title("Sentiment confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### Failure cases

The most informative errors are: positives the model called negative, and negatives the model called positive. These are where the rating signal and the text signal genuinely disagree.

In [ ]:
wrong = eval_df[eval_df["sentiment_true"] != eval_df["sentiment_pred"]]
print(f"Errors: {len(wrong)} / {len(eval_df)} ({len(wrong)/len(eval_df):.1%})")

interesting = wrong[
    ((wrong["sentiment_true"] == "positive") & (wrong["sentiment_pred"] == "negative"))
    | ((wrong["sentiment_true"] == "negative") & (wrong["sentiment_pred"] == "positive"))
][["reviews.rating", "sentiment_true", "sentiment_pred", "reviews.text"]]
interesting.head(10)

## 4. Clusterer evaluation

We embed product-level documents with MiniLM and run KMeans for `k` in `[MIN_CLUSTERS, MAX_CLUSTERS]`. Silhouette score is the headline metric. We also inspect cluster sizes (a degenerate cluster of size 1 is a red flag) and look at sample product names per cluster to confirm the categories are coherent.

In [ ]:
clusterer = ProductClusterer()
product_docs = clusterer.build_product_documents(df)
embeddings = clusterer.embed_reviews(product_docs["cluster_document"].astype(str).tolist())
embeddings.shape

In [ ]:
k_range = list(range(MIN_CLUSTERS, MAX_CLUSTERS + 1))
silhouettes = []
for k in k_range:
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto").fit_predict(embeddings)
    silhouettes.append(silhouette_score(embeddings, labels))

best_k = k_range[int(np.argmax(silhouettes))]
scores_df = pd.DataFrame({"k": k_range, "silhouette": silhouettes})
print(f"Best k by silhouette: {best_k}")
scores_df

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
colors = ["#10b981" if k == best_k else "#94a3b8" for k in k_range]
ax.bar(k_range, silhouettes, color=colors)
ax.set_xlabel("k")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette score per k (selected k highlighted)")
plt.tight_layout()
plt.show()

In [ ]:
# Cluster sizes & sample products at the chosen k.
labels_best = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init="auto").fit_predict(embeddings)
product_docs["category_id"] = labels_best

sizes = product_docs.groupby("category_id")["name"].count().rename("products")
review_volumes = product_docs.groupby("category_id")["review_count"].sum().rename("reviews")
sizes_df = pd.concat([sizes, review_volumes], axis=1)
sizes_df

In [ ]:
# Five sample product names per cluster — eyeball test for category coherence.
for cid, grp in product_docs.groupby("category_id"):
    sample = grp.sort_values("review_count", ascending=False).head(5)["name"].tolist()
    print(f"Cluster {cid}:")
    for n in sample:
        print(f"  - {n}")
    print()

## 5. Summarizer evaluation

The summarizer takes an aggregated insight dict and writes a recommendation article. We evaluate it by:

- Generating an article for one category with FLAN-T5
- Comparing it to the deterministic `_fallback_article` (which is built from the same structured facts) using ROUGE
- Logging length and unique-word ratio as a degeneracy check (the same checks the production code uses to decide between LLM and fallback)

ROUGE here doesn't measure *truth* — both texts are derived from the same facts. It measures **coverage**: does the LLM mention the same products and themes the deterministic article does? A near-zero ROUGE means the LLM is hallucinating or going off-topic.

In [ ]:
from src.aggregation import aggregate_category_insights, name_categories

# Build one real insight to feed the summarizer.
labeled = df.assign(sentiment=df["reviews.rating"].map(sentiment_from_rating))
clustered = clusterer.cluster(labeled)
clustered = name_categories(clustered)
insights = aggregate_category_insights(clustered)
sample_insight = insights[0]
sample_insight["category_name"], len(sample_insight["top_products"])

In [ ]:
writer = RecommendationWriter()
reference = writer._fallback_article(sample_insight)

try:
    generator = load_summarizer()
    candidate = generator(build_safe_prompt(sample_insight))[0]["generated_text"].strip()
except Exception as exc:
    print(f"FLAN-T5 unavailable, evaluating fallback only: {exc}")
    candidate = reference

words = candidate.split()
unique_ratio = len(set(words)) / max(len(words), 1)
print(f"Candidate length: {len(words)} words  |  unique-word ratio: {unique_ratio:.2f}")
print("\n--- candidate (head) ---")
print("\n".join(candidate.splitlines()[:10]))

In [ ]:
# Lightweight ROUGE (no extra dep): n-gram and LCS overlap on whitespace tokens.
def _rouge_n(ref: str, cand: str, n: int) -> dict:
    def ngrams(tokens: list[str]) -> list[tuple[str, ...]]:
        return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

    ref_ng = ngrams(ref.lower().split())
    cand_ng = ngrams(cand.lower().split())
    if not ref_ng or not cand_ng:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    overlap = sum(1 for ng in cand_ng if ng in ref_ng)
    precision = overlap / len(cand_ng)
    recall = overlap / len(ref_ng)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}

def _rouge_l(ref: str, cand: str) -> dict:
    ref_tokens = ref.lower().split()
    cand_tokens = cand.lower().split()
    m, n = len(ref_tokens), len(cand_tokens)
    if m == 0 or n == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i - 1] == cand_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[m][n]
    precision = lcs / n
    recall = lcs / m
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}

rouge_scores = pd.DataFrame({
    "ROUGE-1": _rouge_n(reference, candidate, 1),
    "ROUGE-2": _rouge_n(reference, candidate, 2),
    "ROUGE-L": _rouge_l(reference, candidate),
}).round(3)
rouge_scores

## 6. Failure-case notes

Sentiment classifier — common failure modes seen in section 3:
- 5-star reviews that are sarcastic or list complaints before praise → predicted `negative`.
- 1-2-star reviews that start with "I love the idea, but…" → predicted `positive`.
- 3-star reviews are inherently noisy: many are neither praise nor complaint, just neutral observations. Whether the model gets these right depends mostly on the confidence threshold.

Clusterer — qualitative failure modes:
- A cluster of size 1 means a single product didn't fit anywhere. It's worth checking whether `MIN_CLUSTERS` is too high.
- A cluster mixing tablets and accessories is a sign the embedding lost product-type signal — typically because the product names were near-identical ("AmazonBasics …").

Summarizer — what to watch:
- Length below ~60 words usually means FLAN-T5 truncated early; fallback kicks in.
- Low ROUGE-1 means the LLM dropped the actual product names. That's the worst outcome — the article is no longer recommending the right products.

## 7. Takeaways

Document the headline numbers from section 3, 4, and 5 here so the presentation can quote them directly. Suggested format:

| Model | Metric | Value |
|---|---|---|
| Sentiment classifier | Macro-F1 | _(fill from section 3)_ |
| Sentiment classifier | Accuracy | _(fill from section 3)_ |
| Clusterer | Silhouette @ best k | _(fill from section 4)_ |
| Clusterer | Best k | _(fill from section 4)_ |
| Summarizer | ROUGE-1 F1 | _(fill from section 5)_ |
| Summarizer | ROUGE-L F1 | _(fill from section 5)_ |

Note any iterations made between runs — confidence threshold changes, k range changes, prompt edits — so the evaluation tells a story of progress rather than a single snapshot.